# 在 Notebook 中提问 LightRAG

这个 notebook 用来直接调用项目的 `/chat/stream` 流式接口，不用手写 `curl`。

启动方式：

```bash
python -m agentic_rag.main serve
```

默认 API 地址按 `APP_PORT=8010` 处理，避免和 `.env` 里的 vLLM `8000` 冲突。

In [ ]:
import json
from pathlib import Path
from pprint import pprint

import pandas as pd
import requests

API_BASE = "http://127.0.0.1:8010"
TRIPLES_PATH = Path("../lightrag/standard_triples.json")


def _iter_sse_events(response):
    for raw_line in response.iter_lines(chunk_size=1, decode_unicode=True):
        if not raw_line:
            continue
        if not raw_line.startswith("data: "):
            continue
        payload = raw_line[6:]
        if not payload.strip():
            continue
        yield json.loads(payload)


def ask(question: str, force_route: str = "lightrag"):
    payload = {
        "question": question,
        "force_route": force_route,
    }
    answer_parts = []
    route = None
    reason = None
    with requests.post(
        f"{API_BASE}/chat/stream",
        json=payload,
        stream=True,
        timeout=(10, None),
        headers={"Accept": "text/event-stream"},
    ) as response:
        response.raise_for_status()
        print("回答:\n")
        for event in _iter_sse_events(response):
            event_type = event.get("type")
            if event_type == "ready":
                print("[连接] 已连接到流式接口")
            elif event_type == "meta":
                route = event.get("route")
                reason = event.get("reason")
                print("路由:", route)
                print("原因:", reason)
                print()
            elif event_type == "status":
                print(f"[状态] {event.get('message', '')}")
            elif event_type == "delta":
                chunk = event.get("content", "")
                if chunk:
                    answer_parts.append(chunk)
                    print(chunk, end="", flush=True)
            elif event_type == "error":
                raise RuntimeError(event.get("message") or "流式接口返回错误。")
            elif event_type == "done":
                route = route or event.get("route")
                reason = reason or event.get("reason")
        print()
    answer = "".join(answer_parts).strip()
    return {
        "answer": answer,
        "route": route,
        "reason": reason,
    }


def load_triples(path: Path = TRIPLES_PATH):
    if not path.exists():
        raise FileNotFoundError(f"未找到标准三元组文件: {path.resolve()}")
    return json.loads(path.read_text(encoding="utf-8"))


def triples_df(path: Path = TRIPLES_PATH):
    triples = load_triples(path)
    return pd.DataFrame(triples)


def search_triples(keyword: str, path: Path = TRIPLES_PATH, limit: int = 20):
    df = triples_df(path)
    lowered = keyword.lower()
    mask = (
        df["subject"].str.lower().str.contains(lowered, na=False)
        | df["predicate"].str.lower().str.contains(lowered, na=False)
        | df["object"].str.lower().str.contains(lowered, na=False)
        | df["evidence"].str.lower().str.contains(lowered, na=False)
    )
    return df.loc[mask].head(limit)


In [ ]:
# 先测一下服务是否正常
requests.get(f"{API_BASE}/health", timeout=30).json()

In [ ]:
ask("Who publishes ASME B16.5-2013?")

In [ ]:
ask("What is Table II-9 about in ASME B16.5-2013?")

In [ ]:
ask("What does ASME B16.5-2013 say about ring joint facings?")

## 查看标准三元组

重新执行入库后，项目会在 `lightrag/standard_triples.json` 中写入标准谓语三元组。下面几个单元格用来检查导出结果。

In [ ]:
# 查看文件是否存在，以及总共有多少条三元组
triples = load_triples()
print("标准三元组文件:", TRIPLES_PATH.resolve())
print("三元组总数:", len(triples))

In [ ]:
# 随机看前几条
pprint(triples[:5])

In [ ]:
# 以表格形式查看，便于筛选
df = triples_df()
df.head(10)

In [ ]:
# 按关键词筛选，例如查询 ASME B16.5-2013 的相关三元组
search_triples("ASME B16.5-2013")

In [ ]:
# 查询温度和材料相关的标准三元组
search_triples("Graphite")